## 1. Setup Awal: Import Library dan Definisi Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import re
import string
import os
import pickle
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Unduh resource NLTK yang diperlukan (stopwords, punkt)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Definisi path untuk file input dan output
DRIVE_PATH = '/content/drive/MyDrive/Skripsi Kode/'
OUTPUT_DIR = os.path.join(DRIVE_PATH, 'OutputArtefak/')

# Path input sekarang menunjuk ke file yang berisi 'combined_text'
PROCESSED_INPUT_PATH = os.path.join(OUTPUT_DIR, 'df_data_for_tfidf_training.csv')

# Path metadata asli untuk mengambil info tampilan (title, author, desc)
APP_METADATA_PATH = os.path.join(OUTPUT_DIR, 'df_metadata_for_app.csv')

# Path output untuk artefak TF-IDF
TFIDF_VECTORIZER_PATH = os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.pkl')
TFIDF_MATRIX_PATH = os.path.join(OUTPUT_DIR, 'tfidf_matrix.pkl')
TFIDF_READY_DATA_PATH = os.path.join(OUTPUT_DIR, 'df_tfidf_ready.csv')
TFIDF_RESULTS_PATH = os.path.join(OUTPUT_DIR, 'hasil_pencarian_tfidf.csv')

print("Library dan path telah disiapkan.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Library dan path telah disiapkan.


## 2. Pemuatan dan Pra-pemrosesan Data Spesifik untuk TF-IDF

In [ ]:
# Fungsi pra-pemrosesan teks untuk TF-IDF
def preprocess_text_for_tfidf(text):

    if not isinstance(text, str):
        return ""

    # 1. Konversi ke huruf kecil
    text = text.lower()

    # 2. Hapus semua tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Tokenisasi
    tokens = text.split()

    # 4. Hapus stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # 5. Stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]

    return " ".join(tokens)

In [ ]:
# Memuat data yang berisi 'combined_text' untuk pelatihan TF-IDF
print(f"Memuat data pelatihan dari: {PROCESSED_INPUT_PATH}")
try:
    # Ini adalah DataFrame untuk training
    df_training_tfidf = pd.read_csv(PROCESSED_INPUT_PATH, encoding='utf-8')
    df_training_tfidf.dropna(subset=['combined_text'], inplace=True)
    df_training_tfidf.reset_index(drop=True, inplace=True)
    print("Data pelatihan TF-IDF berhasil dimuat.")
except FileNotFoundError:
    print(f"ERROR: File {PROCESSED_INPUT_PATH} tidak ditemukan.")
    exit()

# Memuat data metadata untuk menampilkan hasil nanti
print(f"Memuat metadata tampilan dari: {APP_METADATA_PATH}")
try:
    # Ini adalah DataFrame untuk lookup informasi tampilan
    df_display_metadata = pd.read_csv(APP_METADATA_PATH, encoding='utf-8')
    print("Metadata tampilan berhasil dimuat.")
except FileNotFoundError:
    print(f"ERROR: File {APP_METADATA_PATH} tidak ditemukan.")
    exit()

# Terapkan fungsi pra-pemrosesan TF-IDF pada kolom 'combined_text' dari data training
print("Memulai pra-pemrosesan teks spesifik untuk TF-IDF...")
df_training_tfidf['processed_text_tfidf'] = df_training_tfidf['combined_text'].apply(preprocess_text_for_tfidf)
print("Pra-pemrosesan TF-IDF selesai.")

Memuat data pelatihan dari: /content/drive/MyDrive/Skripsi Kode/OutputArtefak/df_data_for_tfidf_training.csv
Data pelatihan TF-IDF berhasil dimuat.
Memuat metadata tampilan dari: /content/drive/MyDrive/Skripsi Kode/OutputArtefak/df_metadata_for_app.csv


/tmp/ipython-input-2405164989.py:17: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_display_metadata = pd.read_csv(APP_METADATA_PATH, encoding='utf-8')


Metadata tampilan berhasil dimuat.
Memulai pra-pemrosesan teks spesifik untuk TF-IDF...
Pra-pemrosesan TF-IDF selesai.


In [ ]:
# Menampilkan contoh hasil pra-pemrosesan
print("Contoh Teks Setelah Pra-pemrosesan TF-IDF:")
display(df_training_tfidf[['combined_text', 'processed_text_tfidf']].head())

Contoh Teks Setelah Pra-pemrosesan TF-IDF:


,combined_text,processed_text_tfidf
0,between two fires: american indians in the civ...,two fire american indian civil war laurenc hau...
1,"fashion sourcebook 1920s charlotte fiell,emman...",fashion sourcebook 1920 charlott fiellemmanuel...
2,hungary 56 andy anderson the seminal history a...,hungari 56 andi anderson semin histori analysi...
3,all-american anarchist: joseph a. labadie and ...,allamerican anarchist joseph labadi labor move...
4,les oiseaux gourmands jean leveille aujourd'hu...,le oiseaux gourmand jean leveil aujourdhui loi...


In [ ]:
df_training_tfidf.to_csv(TFIDF_READY_DATA_PATH, index=False)
print(f"[OUTPUT] Data yang siap untuk TF-IDF telah disimpan di:\n{TFIDF_READY_DATA_PATH}")

[OUTPUT] Data yang siap untuk TF-IDF telah disimpan di:
/content/drive/MyDrive/Skripsi Kode/OutputArtefak/df_tfidf_ready.csv


## 3. Pembangunan Model TF-IDF

In [ ]:
# Periksa apakah model sudah ada untuk menghindari proses ulang
if os.path.exists(TFIDF_VECTORIZER_PATH) and os.path.exists(TFIDF_MATRIX_PATH):
    print("\nModel TF-IDF sudah ada. Memuat dari file...")
    with open(TFIDF_VECTORIZER_PATH, 'rb') as f:
        tfidf_vectorizer = pickle.load(f)
    with open(TFIDF_MATRIX_PATH, 'rb') as f:
        corpus_tfidf_matrix = pickle.load(f)
    print("Model dan matriks TF-IDF berhasil dimuat.")
else:
    print("\nMembangun model TF-IDF baru...")
    # Inisialisasi TfidfVectorizer
    tfidf_vectorizer = TfidfVectorizer()

    # Latih (fit) dan transformasikan korpus teks
    corpus_tfidf_matrix = tfidf_vectorizer.fit_transform(df_training_tfidf['processed_text_tfidf'])

    # Simpan model dan matriks yang telah dibangun
    with open(TFIDF_VECTORIZER_PATH, 'wb') as f:
        pickle.dump(tfidf_vectorizer, f)
    with open(TFIDF_MATRIX_PATH, 'wb') as f:
        pickle.dump(corpus_tfidf_matrix, f)
    print("Model dan matriks TF-IDF berhasil dibangun dan disimpan.")

print(f"Dimensi matriks TF-IDF: {corpus_tfidf_matrix.shape}")


Membangun model TF-IDF baru...
Model dan matriks TF-IDF berhasil dibangun dan disimpan.
Dimensi matriks TF-IDF: (100021, 536578)


## 4. Fungsi Pencarian dan Pengujian

In [ ]:
def search_tfidf(query, k=10):

    # Pra-pemrosesan kueri dengan fungsi yang sama
    processed_query = preprocess_text_for_tfidf(query)
    query_vector = tfidf_vectorizer.transform([processed_query])

    # Hitung kemiripan kosinus antara kueri dan semua dokumen
    cosine_similarities = cosine_similarity(query_vector, corpus_tfidf_matrix).flatten()

    processed_query = preprocess_text_for_tfidf(query)
    query_vector = tfidf_vectorizer.transform([processed_query])
    cosine_similarities = cosine_similarity(query_vector, corpus_tfidf_matrix).flatten()

    # Mengurutkan dari skor tertinggi dan ambil k teratas
    top_k_indices = cosine_similarities.argsort()[-k:][::-1]

    # Ambil metadata dari DataFrame tampilan, bukan DataFrame training
    results = df_display_metadata.iloc[top_k_indices].copy()
    results['score'] = cosine_similarities[top_k_indices]

    return results[['book_id', 'title', 'author', 'desc', 'score']]

In [ ]:
# Contoh Pengujian
test_query = "space adventure and galaxy exploration"
print(f"\nMelakukan pencarian TF-IDF untuk kueri: '{test_query}'")

search_results_tfidf = search_tfidf(test_query, k=10)


Melakukan pencarian TF-IDF untuk kueri: 'space adventure and galaxy exploration'


In [ ]:
# Menampilkan hasil dengan format yang lebih rapi
if not search_results_tfidf.empty:
    print("\n--- Hasil Pencarian TF-IDF ---")
    display(search_results_tfidf)

    # Simpan hasil ke file CSV
    search_results_tfidf.to_csv(TFIDF_RESULTS_PATH, index=False)
    print(f"\nHasil pencarian telah disimpan ke: {TFIDF_RESULTS_PATH}")
else:
    print("Tidak ada hasil yang ditemukan untuk kueri tersebut.")


--- Hasil Pencarian TF-IDF ---


,book_id,title,author,desc,score
21026,21026,Phase World,"C.J. Carella,Kevin Siembieda","Phase World, is an incredible transdimensional...",0.359055
5795,5795,Untouched By Human Hands,Robert Sheckley,The 1950s saw publication of Sheckley's 1st fo...,0.349989
22547,22547,Galaxies and How to Observe Them,"Wolfgang Steinicke,Richard Jakiel",This book is a unique work satisfying the need...,0.347669
39096,39096,Space Probes: 50 Years of Exploration from Lun...,"Philippe Seguela,James E. Oberg","\n ,The first complete, up-to-date history of...",0.343021
43776,43776,C-5 Galaxy in Action,Richard Lippincott,C-5 Galaxy first made its appearance in March ...,0.303468
10470,10470,How to Turn a Place Around: A Handbook for Cre...,Project for Public Spaces,"Book by Spaces, Project for Public",0.300040
48023,48023,Codex: Necrons (5th Edition),Matthew Ward,"In the darkness of the ancient past, the Necro...",0.291993
62358,62358,Personal Space Camp,Julia Cook,Teaching Children the Concepts of Personal Spa...,0.291563
83633,83633,The Art of Dead Space,Martin Robinson,"The Art of Dead Space, is the ultimate gallery...",0.289509
1026,1026,Star Drive Campaign Setting,"David Eckelberry,Richard Baker",Welcome to the future's edge with this first c...,0.283831



Hasil pencarian telah disimpan ke: /content/drive/MyDrive/Skripsi Kode/OutputArtefak/hasil_pencarian_tfidf.csv


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=search_results_tfidf)

https://docs.google.com/spreadsheets/d/1eggy4YE_NrlVP9_5wrxBoC6zNXjHvV4NYN_cmVB-HPs/edit#gid=0
